# Gaussian Distribution

**Goal:** Implement the Gaussian pdf and log-pdf from scratch, validate against `torch.distributions.Normal`, recover parameters from samples, and visualise a 2-D isotropic Gaussian.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Gaussian PDF from Scratch

The univariate Gaussian density is:

```
p(x) = 1 / (sigma * sqrt(2*pi)) * exp(-(x - mu)^2 / (2 * sigma^2))
```

Taking the log gives the log-likelihood per sample:
```
log p(x) = -0.5 * log(2*pi) - log(sigma) - (x - mu)^2 / (2 * sigma^2)
```

In [2]:
import math

MU = torch.tensor(3.0, device=device)
SIGMA = torch.tensor(1.5, device=device)


def gaussian_pdf(x: torch.Tensor, mu: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    """Univariate Gaussian pdf: 1/(sigma*sqrt(2*pi)) * exp(-(x-mu)^2 / (2*sigma^2))."""
    norm = 1.0 / (sigma * math.sqrt(2.0 * math.pi))
    return norm * torch.exp(-0.5 * ((x - mu) / sigma) ** 2)


def gaussian_log_pdf(x: torch.Tensor, mu: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    """Log of Gaussian pdf: -0.5*log(2*pi) - log(sigma) - (x-mu)^2/(2*sigma^2)."""
    return -0.5 * math.log(2.0 * math.pi) - torch.log(sigma) - 0.5 * ((x - mu) / sigma) ** 2


x_pts = torch.linspace(-3.0, 9.0, 500, device=device)
pdf_vals = gaussian_pdf(x_pts, MU, SIGMA)

print(f"pdf at mu = {gaussian_pdf(MU, MU, SIGMA).item():.6f}")
print(f"log_pdf at mu = {gaussian_log_pdf(MU, MU, SIGMA).item():.6f}")

pdf at mu = 0.265961
log_pdf at mu = -1.324404


In [3]:
# Validate log_pdf against torch.distributions.Normal
normal_dist = torch.distributions.Normal(MU, SIGMA)
log_pdf_torch = normal_dist.log_prob(x_pts)
log_pdf_scratch = gaussian_log_pdf(x_pts, MU, SIGMA)

assert torch.allclose(log_pdf_scratch, log_pdf_torch, atol=1e-5), (
    f"log_pdf mismatch: max diff = {(log_pdf_scratch - log_pdf_torch).abs().max().item()}"
)
print("log_pdf from scratch matches torch.distributions.Normal.log_prob ✓")

log_pdf from scratch matches torch.distributions.Normal.log_prob ✓


## Sampling, Histogram, and Overlaid PDF

Draw N samples from N(μ, σ²) and overlay the theoretical density.

In [4]:
torch.manual_seed(42)
N = 50_000
samples = torch.randn(N, device=device) * SIGMA + MU

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(samples.cpu().numpy(), bins=80, density=True, alpha=0.5, label="samples")
ax.plot(x_pts.cpu().numpy(), pdf_vals.cpu().numpy(), "r-", lw=2, label=f"N({MU.item()}, {SIGMA.item()}²) pdf")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Gaussian samples vs. theoretical pdf")
ax.legend()
plt.tight_layout()
plt.show()

/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_11985/1111186342.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Recovering μ and σ from Samples (MLE)

The MLE estimates are simply the sample mean and sample standard deviation.

In [5]:
mu_hat = samples.mean()
# MLE sigma uses biased variance (divide by N)
sigma_hat = ((samples - mu_hat) ** 2).mean().sqrt()

print(f"True  μ = {MU.item():.4f},  σ = {SIGMA.item():.4f}")
print(f"MLE   μ̂ = {mu_hat.item():.4f},  σ̂ = {sigma_hat.item():.4f}")

assert torch.allclose(mu_hat, MU, atol=0.05), "mu_hat far from true mu"
assert torch.allclose(sigma_hat, SIGMA, atol=0.05), "sigma_hat far from true sigma"
print("Parameter recovery within tolerance ✓")

True  μ = 3.0000,  σ = 1.5000
MLE   μ̂ = 2.9982,  σ̂ = 1.5017
Parameter recovery within tolerance ✓


## 2-D Isotropic Gaussian

A multivariate Gaussian with diagonal covariance σ²I is called *isotropic*: the uncertainty is equal in every direction.

In [6]:
torch.manual_seed(42)

mu_2d = torch.tensor([1.0, -2.0], device=device)
sigma_2d = torch.tensor(0.8, device=device)

# Sample 5000 points from 2-D isotropic N(mu_2d, sigma_2d^2 * I)
samples_2d = torch.randn(5_000, 2, device=device) * sigma_2d + mu_2d

fig, ax = plt.subplots(figsize=(5, 5))
# Use CPU copy for numpy scatter
s2d_cpu = samples_2d.cpu()
ax.scatter(s2d_cpu[:, 0], s2d_cpu[:, 1], alpha=0.15, s=3)
ax.set_aspect("equal")
ax.set_title(f"2-D Isotropic Gaussian, μ={mu_2d.cpu().tolist()}, σ={sigma_2d.item()}")
ax.set_xlabel("x₁")
ax.set_ylabel("x₂")
plt.tight_layout()
plt.show()

# Verify recovered mean and std
print(f"Sample mean: {samples_2d.mean(dim=0).cpu().tolist()}")
print(f"Sample std:  {samples_2d.std(dim=0).cpu().tolist()}")

Sample mean: [1.005844235420227, -1.998203158378601]
Sample std:  [0.7979822754859924, 0.7997548580169678]


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_11985/1426654626.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Idiomatic PyTorch: `torch.distributions.Normal`

In [7]:
dist = torch.distributions.Normal(loc=MU, scale=SIGMA)

x_test = torch.tensor([MU.item() - SIGMA.item(), MU.item(), MU.item() + SIGMA.item()], device=device)
print("Evaluating N(mu, sigma) at [mu-sigma, mu, mu+sigma]:")
print("  log_prob:", dist.log_prob(x_test).cpu().tolist())
print("  prob:    ", dist.log_prob(x_test).exp().cpu().tolist())

# Entropy in nats: 0.5 * log(2*pi*e*sigma^2)
entropy_theoretical = 0.5 * math.log(2 * math.pi * math.e * SIGMA.item() ** 2)
print(f"Entropy (dist.entropy()) = {dist.entropy().item():.6f}")
print(f"Entropy (formula)        = {entropy_theoretical:.6f}")
assert abs(dist.entropy().item() - entropy_theoretical) < 1e-5
print("Entropy matches theoretical formula ✓")

Evaluating N(mu, sigma) at [mu-sigma, mu, mu+sigma]:
  log_prob: [-1.8244036436080933, -1.3244036436080933, -1.8244036436080933]
  prob:     [0.16131381690502167, 0.26596152782440186, 0.16131381690502167]
Entropy (dist.entropy()) = 1.824404
Entropy (formula)        = 1.824404
Entropy matches theoretical formula ✓


## Takeaways

- **Gaussian pdf** is fully characterised by μ (location) and σ (scale). The log-pdf is a quadratic in x — this quadratic shape is why Gaussian noise leads to squared-error loss.
- **MLE recovery:** sample mean and biased std converge to the true parameters as N → ∞.
- **2-D isotropic Gaussian:** diagonal covariance produces a circular uncertainty cloud. Off-diagonal entries would tilt the ellipse.
- **`torch.distributions.Normal`** provides log_prob, sample, and entropy — used in VAEs, normalizing flows, and any probabilistic model with Gaussian likelihood.